# 19 — Quantum Channels

The last foundation: how a state changes when it is *not* alone. Derive the general quantum evolution — a **channel** — by hand from three things you already own (a unitary, a tensor product, a partial trace), read off its Kraus operators, and see why "decoherence" is nothing but entanglement with an environment you then trace out. Then name the three channels that a calibration sheet is secretly describing — dephasing, amplitude damping, depolarizing — reproduce the $T_1$/$T_2$ curves of `01/15` from them, and prove why $T_2 \le 2T_1$. That closes module 01.

**Prerequisites:** `01/06_gates_as_unitary_evolution` (unitarity; the dictionary promise "the general lossy map is the quantum channel of 01/19"), `01/12_from_states_to_density_matrix` (gates as $U\rho U^\dagger$; coherence = off-diagonal), `01/13_pure_vs_mixed` (purity, $I/2$), `01/14_fidelity_and_trace_distance` (trace distance and Helstrom, for exercise 4), `01/15_tomography_of_a_noisy_qubit` (the idle sweep: coherence $e^{-t/T_2}$, population $1 - e^{-t/T_1}$, the promise of names), `01/16_tensor_and_partial_trace` (the partial trace by blocks; "decoherence is becoming part of something larger"), `01/17_cnot_and_two_qubit_gates` (CNOT as controlled-$X$; the block formula), `01/18_entanglement` (the partial transpose; the Bell state; $I/2$ as a part).

**What you'll be able to do**
- Derive the operator-sum (Kraus) form $\rho \to \sum_k K_k\rho K_k^\dagger$ from "unitary on system + environment, then partial trace", and verify $\sum_k K_k^\dagger K_k = I$ entry by entry.
- Explain *completely* positive — and show, with `01/18`'s partial transpose, a positive map that is not a channel.
- Recognise a stochastic matrix as the diagonal case of a channel and a unitary as the reversible case.
- Build dephasing, amplitude damping and depolarizing by hand, reproduce the `01/15` decay laws, and derive $1/T_2 = 1/2T_1 + 1/T_\varphi$.

You can build states (`01/12`), compare them (`01/14`), combine them (`01/16`), correlate and entangle them (`01/17`–`01/18`). Gates move them reversibly (`01/06`). What moved the idling qubit of `01/15` off the sphere was not a gate — nothing reversible shrinks a Bloch vector. Today you meet the object that did it, and you will find that you already know all its parts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit_ibm_runtime.fake_provider import FakeManilaV2

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum.states import KET_0, KET_1, KET_PLUS
from qot_course.quantum.gates import IDENTITY, CNOT, ry, is_unitary
from qot_course.quantum.density import density_matrix, purity, bloch_vector, maximally_mixed
from qot_course.quantum.composite import tensor, partial_trace, bell_state, apply_channel, depolarizing_channel, is_cptp

rng = np.random.default_rng(0)   # for the random state pairs of exercise 4
viz.use_course_style()

## 1. The general evolution, derived: couple, evolve, forget

Start from what is reversible. A closed system evolves by a unitary: $\rho \to U\rho U^\dagger$ (`01/12`). Now let the qubit $S$ interact with something else — an **environment** $E$, which we take to be one more qubit, starting in $|0\rangle_E$. The *pair* is closed, so the pair evolves by a unitary $U$ on $\mathcal H_S \otimes \mathcal H_E$ (`01/16`). We only ever look at $S$ afterwards, so we trace $E$ out (`01/16` §3):

$$
\rho \;\longmapsto\; \rho' = \mathrm{tr}_E\Big[\,U\,\big(\rho \otimes |0\rangle\langle0|_E\big)\,U^\dagger\,\Big].
$$

That is the whole recipe — couple, evolve, forget — and, for a large enough environment, it is the most general thing that can happen to a quantum system — that converse is Stinespring's theorem (1955), which we quote rather than prove (the references do; for a qubit an environment of at most four levels suffices — the depolarizing channel of section 3 needs all four, the examples below need two). Let's turn it into a formula you can compute with. Write the partial trace as the sandwich of `01/16`, $\mathrm{tr}_E[X] = \sum_k (I\otimes\langle k|)\,X\,(I\otimes|k\rangle)$, and insert $\rho\otimes|0\rangle\langle0| = (I\otimes|0\rangle)\,\rho\,(I\otimes\langle0|)$:

$$
\rho' = \sum_k \underbrace{(I\otimes\langle k|)\,U\,(I\otimes|0\rangle)}_{K_k}\;\rho\;\underbrace{(I\otimes\langle0|)\,U^\dagger\,(I\otimes|k\rangle)}_{K_k^\dagger}
= \sum_k K_k\,\rho\,K_k^\dagger .
$$

The $K_k$ are $2\times2$ matrices on the system alone — the **Kraus operators** — and entry by entry they are sub-matrices of $U$: $(K_k)_{ss'} = U_{(s,k),\,(s',0)}$ — fix the environment's row index to $k$ and its column index to $0$, keep the system's indices free. (With the environment as the *left* factor these would be the $2\times2$ blocks of $U$'s first block-column; with it on the right, as here, they are the same entries read with a stride of $2$ — `01/16`'s index rule $k = 2s + e$.) The sum $\rho \to \sum_k K_k\rho K_k^\dagger$ is the **operator-sum form** of a **quantum channel**. It has three properties, each inherited from the recipe:

1. **Linear** in $\rho$ (everything in the recipe is).
2. **Trace-preserving**: $\mathrm{tr}\,\rho' = \mathrm{tr}\big(\rho\sum_k K_k^\dagger K_k\big)$, and $\sum_k K_k^\dagger K_k = \sum_k (I\otimes\langle0|)U^\dagger(I\otimes|k\rangle)(I\otimes\langle k|)U(I\otimes|0\rangle) = (I\otimes\langle0|)\,U^\dagger U\,(I\otimes|0\rangle) = I$ — the *completeness relation*, which is unitarity read on the environment's $|0\rangle$ slot (the sum over $k$ is the **resolution of the identity**, $\sum_k|k\rangle\langle k| = |0\rangle\langle0| + |1\rangle\langle1| = \mathrm{diag}(1,0) + \mathrm{diag}(0,1) = I_E$ — checked in the code).
3. **Positive** (each $K\rho K^\dagger$ is), and in fact *completely* positive — section 2. (The helper `is_cptp` checks the completeness relation only: for a Kraus list that is all there is to check, complete positivity being automatic.)

Here is the simplest environment that does something: a CNOT with the system as control and the environment as target (`01/17`). The environment "copies" the bit. Reading off the entries $U_{(s,k),(s',0)}$ of CNOT: $K_0 = |0\rangle\langle0|$ (the environment stays $|0\rangle$ exactly when the system is $|0\rangle$) and $K_1 = |1\rangle\langle1|$ (it flips to $|1\rangle$ exactly when the system is $|1\rangle$). So $\rho' = |0\rangle\langle0|\rho|0\rangle\langle0| + |1\rangle\langle1|\rho|1\rangle\langle1| = \mathrm{diag}(\rho_{00}, \rho_{11})$: the channel **keeps the diagonal and erases the coherence** — because the environment now *knows* the bit, and a bit that has been learned elsewhere can no longer interfere. Applied to $|+\rangle$ it gives $I/2$: the endpoint `01/18` promised, reached by a unitary and a partial trace and nothing else. Let's compute it three ways.

In [ ]:
def kraus_from_dilation(U, dim_env=2):
    # K_k = (I (x) <k|) U (I (x) |0>), environment = RIGHT factor (stride-2 sub-matrices, not contiguous blocks)
    # with environment on the right, rows/cols are indexed (s, e) -> 2s + e, so block (k, 0) is U[e=k rows, e=0 cols]
    ks = []
    for k in range(dim_env):
        ks.append(U.reshape(2, dim_env, 2, dim_env)[:, k, :, 0])
    return ks

def kraus_from_dilation_by_hand(U):
    # the same, entry by entry: (K_k)_{s s'} = U_{(s,k), (s',0)}
    ks = []
    for k in range(2):
        K = np.zeros((2, 2), dtype=complex)
        for s in range(2):
            for s_ in range(2):
                K[s, s_] = U[2 * s + k, 2 * s_ + 0]
        ks.append(K)
    return ks

# environment = right factor, system = control: CNOT (control left, target right) is exactly the course constant
U_copy = CNOT
K = kraus_from_dilation_by_hand(U_copy)
print("is_unitary(U):", is_unitary(U_copy))
for k, Kk in enumerate(K):
    print(f"K_{k} =\n{Kk.real.astype(int)}")
print("by hand == reshape:", all(np.allclose(a, b) for a, b in zip(K, kraus_from_dilation(U_copy))))
# read them literally off U: rows with e = k, columns with e = 0 (stride 2)
print("U (real):\n", U_copy.real.astype(int))
print("K_0 == U[[0,2]][:, [0,2]]:", np.allclose(K[0], U_copy[[0, 2]][:, [0, 2]]), "   K_1 == U[[1,3]][:, [0,2]]:", np.allclose(K[1], U_copy[[1, 3]][:, [0, 2]]))
print("resolution of the identity |0><0| + |1><1| = I:", np.allclose(sum(np.outer(e, e) for e in np.eye(2)), np.eye(2)))
print("completeness sum_k K_k^dagger K_k =\n", sum(Kk.conj().T @ Kk for Kk in K).real.astype(int), "  is_cptp:", is_cptp(K))

rho_plus = density_matrix(KET_PLUS)
# way 1: the recipe itself -- couple, evolve, forget
joint = U_copy @ tensor(rho_plus, density_matrix(KET_0)) @ U_copy.conj().T
via_recipe = partial_trace(joint, keep=[0], dims=[2, 2])
# way 2: the Kraus sum by hand; way 3: the helper
via_kraus = sum(Kk @ rho_plus @ Kk.conj().T for Kk in K)
via_helper = apply_channel(rho_plus, K)
print("\n|+> in:\n", np.round(rho_plus.real, 2))
print("out via recipe:\n", np.round(via_recipe.real, 2))
print("recipe == Kraus sum == apply_channel:", np.allclose(via_recipe, via_kraus), np.allclose(via_kraus, via_helper),
      "   == I/2:", np.allclose(via_recipe, maximally_mixed(2)))
print("purity in / out:", round(purity(rho_plus), 2), "/", round(purity(via_recipe), 2))
print("the joint state after U is the Bell state (pure, entangled):", np.allclose(joint, density_matrix(bell_state())))

**Read the output.** The two Kraus operators read off CNOT's entries are $|0\rangle\langle0|$ and $|1\rangle\langle1|$; their completeness sum is $I$ entry by entry, as unitarity guarantees. Pushing $|+\rangle$ through the recipe — tensor with $|0\rangle_E$, conjugate by CNOT, trace $E$ out — gives $\mathrm{diag}(\tfrac12, \tfrac12) = I/2$, and the Kraus sum and the helper agree to the last digit. Purity $1 \to \tfrac12$. And the last line is the point of the whole module arc: *the joint state after the interaction is the Bell state* — pure and entangled. Nothing was lost; the qubit's coherence became correlation with the environment, and a partial trace cannot see correlation (`01/16` §5). Decoherence is entanglement you are not allowed to look at.

## 2. Completely positive — and a positive map that is not a channel

Any map built from the recipe sends density matrices to density matrices. But the recipe guarantees more than that, and the "more" has a name. Suppose the qubit you push through the channel is *half of an entangled pair* (`01/18`): the channel acts on A, nothing happens to B — the map is $\Lambda \otimes \mathrm{id}$. Physically the output must still be a valid state. A map that stays valid even when applied to one part of a larger system is **completely positive** (CP); a map that only stays valid on its own system is merely *positive*. Channels are CP — in Kraus form, $(\Lambda\otimes\mathrm{id})(\rho_{AB}) = \sum_k (K_k\otimes I)\rho_{AB}(K_k\otimes I)^\dagger$, a sum of sandwiches, hence positive — and, conversely, every CP trace-preserving map has a Kraus form — Choi's theorem (1975; Kraus 1983), quoted here, proved in the references. Hence **CPTP** = channel.

You have already met a map that is positive but *not* completely positive: the **transpose**. On one qubit, $\rho \to \rho^{\mathsf T}$ keeps eigenvalues, hence maps states to states. Applied to half a Bell pair it is `01/18`'s partial transpose, and the output had the eigenvalue $-\tfrac12$ — not a state. So the transpose is not a channel, no environment can implement it, and the very test you used to *certify* entanglement is the standard example of why "positive" is not enough. Let's see both on the Bell state, and add the classical case: a **stochastic matrix** $T$ (columns = probability vectors) is the channel with Kraus operators $K_{ij} = \sqrt{T_{ij}}\,|i\rangle\langle j|$ — it maps $\mathrm{diag}(p)$ to $\mathrm{diag}(Tp)$ (and erases coherence along the way), which is how the dictionary row "stochastic matrix ↔ channel" is earned.

In [ ]:
rho_bell = density_matrix(bell_state())

def apply_on_A(kraus, rho4):
    return sum(tensor(Kk, IDENTITY) @ rho4 @ tensor(Kk, IDENTITY).conj().T for Kk in kraus)

def partial_transpose_A(rho4):      # transpose the A-indices: (rho^{T_A})_{ab,a'b'} = rho_{a'b,ab'}
    return rho4.reshape(2, 2, 2, 2).transpose(2, 1, 0, 3).reshape(4, 4)

out_dephase = apply_on_A(K, rho_bell)                      # the copy channel on half a Bell pair
out_transpose = partial_transpose_A(rho_bell)               # the transpose map on half a Bell pair
for name, m in [("copy channel (x) id on Bell", out_dephase), ("transpose (x) id on Bell", out_transpose)]:
    ev = np.linalg.eigvalsh(m).real
    ev = np.where(np.abs(ev) < 1e-9, 0.0, ev)
    print(f"{name:32s} eigenvalues {np.round(ev, 3)}   valid state: {ev.min() >= -1e-9}")
print("copy (x) id on Bell == the correlated coins of 01/16:",
      np.allclose(out_dephase, 0.5 * density_matrix(tensor(KET_0, KET_0)) + 0.5 * density_matrix(tensor(KET_1, KET_1))))

# the classical case: a stochastic matrix as a channel
T = np.array([[0.9, 0.3],
              [0.1, 0.7]])                                  # columns sum to 1: P(i <- j)
K_T = [np.sqrt(T[i, j]) * np.outer(np.eye(2)[i], np.eye(2)[j]) for i in range(2) for j in range(2)]
print("\nstochastic channel: is_cptp:", is_cptp(K_T))
p = np.array([0.6, 0.4])
print("T p            =", T @ p)
print("channel(diag p)=", np.round(np.diag(apply_channel(np.diag(p).astype(complex), K_T)).real, 4))
print("on |+> (coherence erased, diagonal moved):\n", np.round(apply_channel(rho_plus, K_T).real, 3))

**Read the output.** The copy channel applied to half a Bell pair gives eigenvalues $(\tfrac12, \tfrac12, 0, 0)$ — a valid state, and in fact exactly the correlated coins of `01/16` §5: dephasing one half of a Bell pair turns quantum correlation into a shared classical coin. The transpose applied to the same half gives $-\tfrac12$: not a state, so not a channel. Same input, same "positive on one qubit" credentials, opposite verdicts — that is what *completely* adds. And the stochastic matrix $T$, packaged as four Kraus operators $\sqrt{T_{ij}}|i\rangle\langle j|$, passes `is_cptp`, sends $\mathrm{diag}(p)$ to $\mathrm{diag}(Tp)$ exactly, and wipes the coherence of $|+\rangle$: classical Markov evolution is the channel that lives on the diagonal. The dictionary row of `01/06` — "the general lossy map is the quantum channel" — is now earned on both sides.

## 3. The three channels on the calibration sheet

`01/15` watched an idling qubit: coherence fell as $e^{-t/T_2}$, population relaxed toward $|0\rangle$ as $1 - e^{-t/T_1}$, and it promised names. Here they are, each built by hand from Kraus operators you can read (Nielsen & Chuang §8.3).

**Dephasing** (phase damping): the environment learns the bit *partially* — a controlled rotation by $\theta$ on $E$ instead of a full CNOT gives $K_0 = \mathrm{diag}(1, \cos\tfrac\theta2)$, $K_1 = \mathrm{diag}(0, \sin\tfrac\theta2)$ (exercise 1 derives them), so $\rho_{01} \to \cos\tfrac\theta2\,\rho_{01}$ and the diagonal is untouched. Writing the surviving fraction as $\sqrt{1-\lambda}$: $K_0 = \mathrm{diag}(1, \sqrt{1-\lambda})$, $K_1 = \mathrm{diag}(0, \sqrt\lambda)$. In time, $\sqrt{1 - \lambda(t)} = e^{-t/T_\varphi}$ — a *pure* dephasing time $T_\varphi$.

**Amplitude damping**: the excited state leaks energy into the environment — $|1\rangle_S|0\rangle_E \to \sqrt{1-\gamma}\,|1\rangle|0\rangle + \sqrt\gamma\,|0\rangle|1\rangle$ — giving $K_0 = \mathrm{diag}(1, \sqrt{1-\gamma})$ and $K_1 = \sqrt\gamma\,|0\rangle\langle1|$. Entry by entry: $\rho_{11} \to (1-\gamma)\rho_{11}$ (population relaxes to $|0\rangle$), $\rho_{01} \to \sqrt{1-\gamma}\,\rho_{01}$ (coherence shrinks by the *square root*). In time, $1 - \gamma(t) = e^{-t/T_1}$.

**Depolarizing**: with probability $p$ replace the state by $I/2$ — $\rho \to (1-p)\rho + p\,I/2$, Kraus $\sqrt{1 - 3p/4}\,I$ and $\sqrt{p/4}\,X, Y, Z$ (exercise 3 checks the identity $\tfrac14(\rho + X\rho X + Y\rho Y + Z\rho Z) = I/2$). Purity of a pure input: $\tfrac12\big(1 + (1-p)^2\big)$, from $1$ to $\tfrac12$ (in general $\tfrac12(1 + (1-p)^2|\vec r|^2)$).

Let's build the three channels and check each entry-by-entry claim.

In [ ]:
def dephasing(lmbda):
    return [np.diag([1, np.sqrt(1 - lmbda)]).astype(complex), np.diag([0, np.sqrt(lmbda)]).astype(complex)]

def amplitude_damping(gamma):
    return [np.diag([1, np.sqrt(1 - gamma)]).astype(complex), np.sqrt(gamma) * np.outer(KET_0, KET_1.conj())]

# dephasing from a dilation: controlled-R_y(theta) on the environment (system = control), blocks read off
theta = 1.1
CRy = tensor(np.outer(KET_0, KET_0.conj()), IDENTITY) + tensor(np.outer(KET_1, KET_1.conj()), ry(theta))
K_dil = kraus_from_dilation_by_hand(CRy)
print("controlled-R_y dilation gives K_0 = diag(1, cos t/2), K_1 = diag(0, sin t/2):",
      np.allclose(K_dil[0], np.diag([1, np.cos(theta / 2)])), np.allclose(K_dil[1], np.diag([0, np.sin(theta / 2)])),
      "  == dephasing(lambda = sin^2 t/2):", all(np.allclose(a, b) for a, b in zip(K_dil, dephasing(np.sin(theta / 2) ** 2))))

rho_t = density_matrix(np.array([np.cos(0.35), np.exp(0.4j) * np.sin(0.35)]))    # a tilted state with a complex coherence
lam, gam, pdep = 0.36, 0.25, 0.3
for name, ks in [("dephasing(0.36)", dephasing(lam)), ("amplitude_damping(0.25)", amplitude_damping(gam)), ("depolarizing(0.3)", depolarizing_channel(pdep))]:
    print(f"\n{name:24s} is_cptp: {is_cptp(ks)}")
    out = apply_channel(rho_t, ks)
    print(f"  rho_11: {rho_t[1,1].real:.4f} -> {out[1,1].real:.4f}   rho_01: {rho_t[0,1]:.4f} -> {out[0,1]:.4f}")
print("\nchecks against the entry-by-entry claims:")
out = apply_channel(rho_t, dephasing(lam));         print("  dephasing:  rho_01 * sqrt(1-lambda) ==", np.isclose(out[0, 1], rho_t[0, 1] * np.sqrt(1 - lam)), "  diagonal untouched:", np.isclose(out[1, 1], rho_t[1, 1]))
out = apply_channel(rho_t, amplitude_damping(gam)); print("  amp. damp.: rho_11 * (1-gamma) ==", np.isclose(out[1, 1], rho_t[1, 1] * (1 - gam)), "  rho_01 * sqrt(1-gamma) ==", np.isclose(out[0, 1], rho_t[0, 1] * np.sqrt(1 - gam)))
out = apply_channel(rho_t, depolarizing_channel(pdep)); print("  depolarizing: (1-p) rho + p I/2 ==", np.allclose(out, (1 - pdep) * rho_t + pdep * np.eye(2) / 2), "  purity (1+(1-p)^2)/2 ==", np.isclose(purity(out), (1 + (1 - pdep) ** 2) / 2))

**Read the output.** The controlled-$R_y$ dilation hands back exactly $\mathrm{diag}(1, \cos\tfrac\theta2)$ and $\mathrm{diag}(0, \sin\tfrac\theta2)$ — dephasing *is* "the environment learns a fraction of the bit", and $\lambda = \sin^2\tfrac\theta2$ is how much it learned (a full CNOT, $\theta = \pi$, is $\lambda = 1$: section 1). All three channels pass `is_cptp`. On a tilted state with a complex coherence, dephasing leaves the diagonal alone and scales $\rho_{01}$ by $\sqrt{1-\lambda}$; amplitude damping scales $\rho_{11}$ by $1-\gamma$ and $\rho_{01}$ by $\sqrt{1-\gamma}$ — the square root that will make $T_2 \le 2T_1$; depolarizing is $(1-p)\rho + pI/2$ with the purity closed form. Each claim in the prose is a `True` in the output.

### Two channels, one $T_2$ — and why $T_2 \le 2T_1$

Now put the first two together, as a real qubit does. Coherence picks up *both* factors: $\rho_{01}(t) = \rho_{01}(0)\,e^{-t/2T_1}\,e^{-t/T_\varphi}$. The measured $T_2$ is defined by $\rho_{01}(t) \propto e^{-t/T_2}$, so

$$
\frac{1}{T_2} = \frac{1}{2T_1} + \frac{1}{T_\varphi} \qquad\Longrightarrow\qquad T_2 \le 2T_1 ,
$$

with equality only when there is no pure dephasing at all. That is the law `01/15` left as a promise: energy relaxation *alone* already costs coherence at half its own rate, and dephasing only adds to it. Let's run the composed channel with the calibration sheet's $T_1$, $T_2$ of the qubit `01/15` used and lay it over the `01/15` laws.

In [ ]:
fake = FakeManilaV2()
PHYS = 3                                                     # the qubit 01/15 used
q = fake.target.qubit_properties[PHYS]
T1, T2 = q.t1, q.t2
T_phi = 1 / (1 / T2 - 1 / (2 * T1))                          # pure dephasing time implied by the sheet
print(f"qubit {PHYS}: T1 = {T1*1e6:.1f} us, T2 = {T2*1e6:.1f} us  ->  2 T1 = {2*T1*1e6:.1f} us >= T2: {T2 <= 2*T1},   T_phi = {T_phi*1e6:.1f} us")

def idle_channel(t):
    # amplitude damping for time t, then pure dephasing for time t (they commute on populations/coherence)
    gamma = 1 - np.exp(-t / T1)
    lam = 1 - np.exp(-2 * t / T_phi)                         # sqrt(1 - lam) = e^{-t/T_phi}
    return amplitude_damping(gamma), dephasing(lam)

t_us = np.linspace(0, 150, 151)
X_out, Z_out, pur = [], [], []
for t in t_us * 1e-6:
    kA, kD = idle_channel(t)
    rho = apply_channel(apply_channel(rho_plus, kA), kD)
    r = bloch_vector(rho)
    X_out.append(r[0]); Z_out.append(r[2]); pur.append(purity(rho))
X_out, Z_out, pur = map(np.array, (X_out, Z_out, pur))
x_law = np.exp(-t_us * 1e-6 / T2)                            # the 01/15 laws (meter factor 1 here: no readout error)
z_law = 1 - np.exp(-t_us * 1e-6 / T1)
print("channel <X>(t) == e^{-t/T2}:", np.allclose(X_out, x_law), "   channel <Z>(t) == 1 - e^{-t/T1}:", np.allclose(Z_out, z_law))
print(f"minimum purity along the idle: {pur.min():.3f} at t = {t_us[pur.argmin()]:.0f} us")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
ax = axes[0]
ax.plot(t_us, X_out, color=COLORS["quantum"], lw=2.5, label=r"$\langle X\rangle$ from the channel (coherence)")
ax.plot(t_us[::10], x_law[::10], "o", color=COLORS["quantum"], ms=6, mfc="white", label=r"$e^{-t/T_2}$  (01/15 law)")
ax.plot(t_us, Z_out, color=COLORS["highlight"], lw=2.5, label=r"$\langle Z\rangle$ from the channel (population)")
ax.plot(t_us[::10], z_law[::10], "s", color=COLORS["highlight"], ms=6, mfc="white", label=r"$1 - e^{-t/T_1}$  (01/15 law)")
ax.plot(t_us, np.exp(-t_us * 1e-6 / (2 * T1)), color=COLORS["muted"], lw=1.2, ls=":", label=r"$e^{-t/2T_1}$: coherence from $T_1$ alone")
ax.set_xlabel("idle time (us)"); ax.set_ylabel("Pauli average"); ax.set_title("Amplitude damping + dephasing\nreproduce the 01/15 idle laws", pad=12)
ax.legend(fontsize=8, loc="upper right")
ax = axes[1]
ax.plot(X_out, Z_out, color=COLORS["flow"], lw=2.5)
ax.scatter([X_out[0]], [Z_out[0]], color=COLORS["source"], zorder=5, label=r"$t = 0$: $|+\rangle$")
ax.scatter([X_out[-1]], [Z_out[-1]], color=COLORS["target"], zorder=5, label=r"$t = 150\,\mu$s: toward $|0\rangle$")
circle = plt.Circle((0, 0), 1, fill=False, color=COLORS["grid"], lw=1.5)
ax.add_patch(circle); ax.set_aspect("equal"); ax.set_xlim(-1.1, 1.1); ax.set_ylim(-1.1, 1.1)
ax.set_xlabel(r"$\langle X\rangle$"); ax.set_ylabel(r"$\langle Z\rangle$"); ax.set_title("The Bloch vector's path in the x–z plane:\ninto the ball, then up toward the pole", pad=12)
ax.legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

**Read the output and the figure.** The calibration sheet of qubit $3$ says $T_2 \approx 54\,\mu$s against $2T_1 \approx 358\,\mu$s — comfortably inside the law, with a pure dephasing time $T_\varphi \approx 64\,\mu$s doing most of the damage. Composing amplitude damping (with $\gamma = 1 - e^{-t/T_1}$) and dephasing (with $\sqrt{1-\lambda} = e^{-t/T_\varphi}$) reproduces *exactly* the two laws `01/15` fitted to tomography data — `allclose` on both — with no fitting here: the curves are the channels. The dotted line shows what coherence would do under $T_1$ alone ($e^{-t/2T_1}$, slow); the gap to the solid line is $T_\varphi$. On the right, the Bloch vector leaves the sphere along the $x$-axis (coherence dies), dips to purity $\approx 0.59$ at $\approx 73\,\mu$s — `01/15` read $\approx 0.58$ for the same dip *through its meter* (the factor $1 - 2\epsilon$) — and climbs toward the north pole $|0\rangle$ as the population relaxes. The idle sweep of `01/15` was not a mystery of the hardware; it was two named channels, and you can now write their Kraus operators from memory.

## Your turn

1. **Dephasing, derived.** Write the $4\times4$ of the controlled-$R_y(\theta)$ (system = control, environment = target) by the block formula of `01/16`, read off $(K_k)_{ss'} = U_{(s,k),(s',0)}$ by hand, and show $K_0 = \mathrm{diag}(1, \cos\tfrac\theta2)$, $K_1 = \mathrm{diag}(0, \sin\tfrac\theta2)$; check $K_0^\dagger K_0 + K_1^\dagger K_1 = I$ entry by entry. Then show that dephasing can also be written as *measure $Z$ and forget the result*: $\rho \to (1 - q)\rho + q\,Z\rho Z$ for the right $q(\lambda)$ — two different Kraus sets, one channel (Kraus operators are not unique).
2. **Noise on half a Bell pair — the right meter.** Push one qubit of the Bell state through `depolarizing_channel(p)` for $p \in [0, 1]$ (`apply_on_A`). Plot the **most negative eigenvalue of the partial transpose** (`01/18`) against $p$: at which $p$ does the pair stop being entangled? (Show by hand that the output is the Werner state of `01/18` exercise 4 with weight $1 - p$, so the threshold is $p = 2/3$.) Now also plot `entanglement_entropy` (`from qot_course.quantum.composite import entanglement_entropy`) of the output against $p$ — it stays at exactly $1$ bit throughout, because $\rho_B = I/2$ whatever happens to A. Explain in one sentence why the second curve says *nothing* here (`01/18` §3: the global state is no longer pure).
3. **Depolarizing from the Paulis.** Prove by hand that $\tfrac14\big(\rho + X\rho X + Y\rho Y + Z\rho Z\big) = \tfrac{I}{2}$ for every qubit $\rho$ (write $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ from `01/12` and use $X Y X = -Y$ etc.), and hence that the Kraus set $\{\sqrt{1-3p/4}\,I, \sqrt{p/4}\,X, \sqrt{p/4}\,Y, \sqrt{p/4}\,Z\}$ implements $(1-p)\rho + pI/2$. Where does $p = 4/3$ send the Bloch vector, and why does the Kraus set stop existing there (the weight $1 - 3p/4$ turns negative — `is_cptp` at $p = 1.4$ fails only because of a square root of a negative number; it is not a CP test)? For $p > 4/3$ apply the map $(1-p)\rho + pI/2$ to half a Bell pair, as in section 2, and read the eigenvalues. Then a fourth Pauli channel: the **bit-flip** $\{\sqrt{1-p}\,I, \sqrt p\,X\}$ — show it leaves $|+\rangle$ and $|-\rangle$ untouched and shrinks the $y$–$z$ components of the Bloch vector by $1 - 2p$ (use $XYX = -Y$, $XZX = -Z$).
4. **Harder: what a channel cannot do.** Prove from the Kraus form that a channel *contracts* the trace distance (`01/14`): $T(\Lambda\rho, \Lambda\sigma) \le T(\rho, \sigma)$. Hint: $\rho - \sigma$ is Hermitian and traceless; split its eigen-decomposition into positive and negative parts, $\rho - \sigma = P - Q$ with $P, Q \ge 0$ (for a qubit, eigenvalues $\pm T$ as in `01/14`), so $T = \mathrm{tr}P = \mathrm{tr}Q$; use that $\|A\|_1 = \mathrm{tr}A$ for $A \ge 0$, the triangle inequality for $\|\cdot\|_1$, and that $\Lambda$ is positive and trace-preserving ($\mathrm{tr}\,\Lambda P = \mathrm{tr}P$). Check it numerically on random pairs of qubit states (`rng`) for the three channels of section 3. Finally, argue (two sentences) why that contraction is *the* reason no measurement downstream of noise can recover what the noise erased — the honest accounting behind the hardware floor of module `05`.

## What you built

- You derived the operator-sum form from "couple, evolve, forget": $K_k = \langle k|_E\,U\,|0\rangle_E$ are sub-matrices of $U$ read entry by entry, $\sum_k K_k^\dagger K_k = I$ is unitarity read on the environment's $|0\rangle$ slot, and the copy channel (CNOT with an environment) turns $|+\rangle$ into $I/2$ while the joint state is the pure Bell state — decoherence is entanglement you then trace out.
- You saw what *completely* positive adds: the copy channel on half a Bell pair gives the coins of `01/16`; the transpose on half a Bell pair gives `01/18`'s $-\tfrac12$ — positive, not a channel. You earned the stochastic-matrix row: Markov evolution is the channel that lives on the diagonal.
- You built dephasing, amplitude damping and depolarizing from their Kraus operators, checked every entry-by-entry claim, derived $1/T_2 = 1/2T_1 + 1/T_\varphi$, and reproduced the `01/15` idle laws from the calibration sheet with no fitting.

**Module 01 complete.** Across nineteen notebooks you went from two amplitudes to the Bloch sphere, measurement, gates, expectation values and shot noise; to the density matrix, mixedness, distances and tomography on a noisy device; to two systems, the partial trace, CNOT, entanglement and now channels — the full language of quantum states, each piece earned by hand. That is a substantial body of understanding, and it is all yours. Take a moment to appreciate how far you have come.

**Dictionary — the whole of module 01** (the table from `01/18`, plus today's row; one forward row remains, for `04/04`). Read it top to bottom once: the left column is the probability theory you knew, the right column is what each piece became.

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| overlap of two probability vectors, $\sum_x \sqrt{p_x q_x}$ (Bhattacharyya coefficient) | **inner product** $\langle\varphi|\psi\rangle$ — the same sum with amplitudes in place of $\sqrt{p_x}$; its modulus squared is the yes-probability of "are you $|\varphi\rangle$?" (fidelity, `01/14`) |
| a reversible operation on a bit is a permutation (identity, NOT); a general operation on probability vectors is a stochastic matrix, usually lossy | a **gate** is a **unitary** matrix $U$ ($U^\dagger U = I$): preserves every inner product, is undone by $U^\dagger$, and rotates the Bloch sphere rigidly (half-angle in the matrix, full angle on the sphere); the general, lossy map is the quantum channel of `01/19` |
| a quantity is a function of the bit, $f(0), f(1)$; its average is $\sum_x p_x f(x)$ | an **observable** is a Hermitian matrix $A = A^\dagger$; its average is $\langle\psi|A|\psi\rangle$ (real), the outcomes are its eigenvalues; the Paulis are gates *and* observables, and $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ is the Bloch vector (measured shot by shot in `01/10`) |
| there is one way to read a bit: look at it | a **measurement basis** is any orthonormal pair $\{|b_0\rangle,|b_1\rangle\}$, $P(k) = |\langle b_k|\psi\rangle|^2$; the hardware natively reads only $Z$, so every other basis is reached by *rotate, then read* — $H$ for $X$, $S^\dagger$ then $H$ for $Y$ (the recombination and $Y$-type rules of `01/02`) |
| a bit has no continuous knob: you flip it or you do not | **rotation gates** $R_x, R_y, R_z$ (and phase gates $P(\lambda)$) are continuous dials — angle $\theta$ in the matrix as $\theta/2$, full $\theta$ on the sphere; any pure state is $R_z(\phi)R_y(\theta)|0\rangle$, any single-qubit gate is $e^{i\alpha}R_z R_y R_z$ (Nielsen & Chuang Thm 4.1) |
| the average $\sum_x p_x f(x)$ is estimated from $N$ samples with a standard error $\sqrt{\mathrm{Var}(f)/N}$ | the **expectation value** $\langle A\rangle = \langle\psi|A|\psi\rangle = \sum_k \lambda_k P(k)$ is the average of the eigenvalue outcomes; from $N$ shots of a $\pm1$ observable it is estimated with standard error $\sqrt{(1 - \langle A\rangle^2)/N} \le 1/\sqrt N$; three Pauli averages determine the state (tomography, `01/15`) |
| a randomized program on bits: initialize, apply logic gates in order, read the output; run it many times and estimate the output frequencies, with sampling error | a **circuit**: qubits initialized to $|0\rangle$, unitary gates in time order, measurement into classical bits; $N$ **shots** give **counts**, a sample of the Born shadow with the error bar above; the device runs a compiled **native** version ($R_z$, $\sqrt X$, $X$, one fixed entangler), equal to yours up to a global phase; a *systematic* miss that does not shrink with $N$ is the signature of noise, and needs $\rho$ (`01/12`) |
| probability vector $p = (p_0, p_1)$: non-negative entries summing to $1$; averages are $\sum_x p_x f(x)$; a fair coin is $(\tfrac12,\tfrac12)$ | **density matrix** $\rho = |\psi\rangle\langle\psi|$ (pure case): Hermitian, unit trace, positive; diagonal = the Born probabilities $p$, off-diagonal = the **coherence** $\tfrac12\sin\theta\,e^{-i\phi}$ (modulus: how much superposition; phase: the relative phase); $P(b) = \langle b|\rho|b\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, gates act as $U\rho U^\dagger$; $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ packs the Bloch vector; a global phase drops out; a $\rho$ that is diagonal in a shared basis *is* a classical $p$ — and the mixtures of `01/13` are the $\rho$'s that are no vector's outer product |
| not knowing which bit you hold is *the only* kind of uncertainty: a coin $p$; a fair coin $(\tfrac12,\tfrac12)$ is as uncertain as it gets, and a mixture of coins is a coin | a **mixed state** $\rho = \sum_i p_i|\psi_i\rangle\langle\psi_i|$ is the *second* kind of uncertainty (ignorance of which state), distinct from superposition (a definite state with random outcomes); its Bloch vector is the average of the parts' — inside the sphere, $I/2$ at the centre; **purity** $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2) \in [\tfrac12, 1]$, eigenvalues $\tfrac12(1 \pm |\vec r|)$, von Neumann entropy = entropy of the eigenvalue coin (preview of `02/01`, `02/09`); a mixture forgets how it was prepared; **the seed:** $|+\rangle$ and $I/2$ share the diagonal $(\tfrac12,\tfrac12)$ and differ only in the corners |
| distances between probability vectors: **total variation** $\tfrac12\sum_x|p_x - q_x|$ (the best one-shot guess between $p$ and $q$ succeeds with probability $\tfrac12(1 + \mathrm{TV})$) and the **Bhattacharyya overlap** $\sum_x\sqrt{p_x q_x}$ | **trace distance** $T = \tfrac12\|\rho - \sigma\|_1 = \tfrac12|\vec r - \vec s|$ for qubits — half the distance in the Bloch **ball** (surface = pure, depth = mixedness); Helstrom: best one-shot guess succeeds with probability $\tfrac12(1 + T)$, by measuring the eigenbasis of $\rho - \sigma$; **fidelity** $F = (\mathrm{tr}\sqrt{\sqrt\rho\sigma\sqrt\rho})^2$ — *this course's convention*: $F = |\langle\varphi|\psi\rangle|^2 = \tfrac12(1 + \vec r\cdot\vec s)$ for pure states, $\langle\psi|\sigma|\psi\rangle$ for one pure, (Bhattacharyya)$^2$ for diagonal states; $F_U = \sqrt F$ is the Uhlmann fidelity of the Bures distance (`02/12`); $1 - \sqrt F \le T \le \sqrt{1 - F}$ |
| estimating a probability vector from data: count frequencies, quote $\pm\sqrt{p(1-p)/N}$; a miscalibrated counter biases every frequency by the same factor, and no number of samples fixes it — you calibrate the counter | **state tomography**: estimate $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ from three circuits' counts, each $\pm\sqrt{(1 - \langle\sigma\rangle^2)/N}$, and pack $\hat\rho = \tfrac12(I + \hat{\vec r}\cdot\vec\sigma)$ (project back onto the ball if noise pushes it out); purity, fidelity and trace distance read straight off $\hat\rho$; tomography reports the state *as seen through the meter* — a symmetric misassignment $\epsilon$ shrinks $\hat{\vec r}$ by $1 - 2\epsilon$, a lopsided one also shifts it (calibrate it away — exercise 2; `05/02` meets the same floor on a QPU) — while an idling qubit's *state* really does decohere, coherence as $e^{-t/T_2}$, population as $1 - e^{-t/T_1}$ (channels, `01/19`) |
| **marginal** of a joint table $p(a,b)$: sum over the other variable, $p_A(a) = \sum_b p(a,b)$ (row sums); independent variables factor, $p = p_A \otimes p_B$, and each marginal is the factor; *correlated* variables (the two matched coins) have perfectly random marginals although the pair is not random at all | two systems live on $\mathcal H_A \otimes \mathcal H_B$ (Kronecker product, index $k = 2a + b$, left factor first); a **product state** $\rho_A \otimes \rho_B$ is the independent case; the **partial trace** $\rho_A = \mathrm{tr}_B\,\rho_{AB}$, $(\rho_A)_{aa'} = \sum_b \rho_{ab,a'b}$ (replace each $2\times2$ block by its trace), is the only map that reproduces every local statistic $\mathrm{tr}(\rho_{AB}(M \otimes I)) = \mathrm{tr}(\rho_A M)$; a diagonal $\rho_{AB}$ reduces *exactly* to the row sums; a correlated joint state has **mixed** marginals — the *third* way a state can be mixed: it is part of something larger (`01/18` makes the whole pure and the parts still mixed) |
| **conditional logic** on two bits: controlled-NOT / XOR, $(a, b) \mapsto (a, a \oplus b)$ — a reversible *permutation* of the four outcomes; on a joint probability table it permutes the four entries; two independent operations, one per bit, can never make the bits agree when they did not before | **CNOT**, the $4\times4$ permutation unitary $|0\rangle\langle0|\otimes I + |1\rangle\langle1|\otimes X$ (controlled-$U$ pattern; control = left factor in this course, Qiskit's `cx(1, 0)`); it permutes the four basis vectors and, by linearity, acts on *superpositions* of the control: $\mathrm{CNOT}\,(\alpha|0\rangle + \beta|1\rangle)|0\rangle = \alpha|00\rangle + \beta|11\rangle$ — a product in, a non-product out (amplitude-matrix determinant $\ne 0$); no $M \otimes N$ can do that; CNOT$(H \otimes I)|00\rangle$ is the Bell state, whose corners are *coherence* between $|00\rangle$ and $|11\rangle$ (a product $|{+}{+}\rangle$ has them too; the coins have none) — entanglement proper is `01/18` |
| a joint table whose whole is *certain* (one outcome, probability $1$) has certain marginals; mixed marginals force a mixed whole (the coins); the correlation of two bits is fully described by their joint table, and any table is a mixture of point masses — every classical correlation is a *shared randomness* | **entanglement**: a *pure* joint state that is not a product (amplitude-matrix $\det\Psi \ne 0$) has **mixed parts** — pure whole, mixed parts, impossible for any table; measured by the **entanglement entropy** $S(\rho_A) = -\sum_i \lambda_i\log_2\lambda_i$ (eigenvalues of the part; $= S(\rho_B)$; $1$ bit for a Bell state; invariant under local gates $U\otimes V$) — **valid only when the whole is pure**: the coins have $S(\rho_A) = 1$ bit and no entanglement; a mixed state is **separable** if it is a mixture of products, and for two qubits the **partial transpose** decides it (a negative eigenvalue = entangled: Bell $-\tfrac12$, coins none); Bell and coins share every $Z$-statistic and differ in $X$: $\langle X\otimes X\rangle = 1$ vs $0$ |
| the general evolution of a probability vector is a **stochastic matrix** (Markov kernel) $p \to Tp$: columns are probability vectors, so probabilities stay non-negative and sum to $1$; a permutation is the reversible case; noise = a coin you did not see being tossed | a **quantum channel**: the general evolution $\rho \to \sum_k K_k\rho K_k^\dagger$ (Kraus operators, $\sum_k K_k^\dagger K_k = I$) — *derived* as "couple to an environment, evolve unitarily, trace the environment out" ($K_k = \langle k|_E\,U\,|0\rangle_E$); linear, trace-preserving and **completely** positive (the transpose is positive but sends half a Bell pair to a matrix with a $-\tfrac12$ eigenvalue — not a channel); a unitary is the reversible case ($K = U$), a stochastic matrix is the diagonal case ($K_{ij} = \sqrt{T_{ij}}\,|i\rangle\langle j|$); **decoherence = entanglement with an environment, then a partial trace**: dephasing (the environment learns the bit; coherence $\times e^{-t/T_\varphi}$), amplitude damping (energy leaks out; population $\times e^{-t/T_1}$, coherence $\times e^{-t/2T_1}$), depolarizing ($\rho \to (1-p)\rho + p\,I/2$); together they give $1/T_2 = 1/2T_1 + 1/T_\varphi$, hence $T_2 \le 2T_1$ — the laws `01/15` fitted |
| transport plan | quantum coupling |

(The last row is still forward-looking — earned in `04/04`.)

## What's next

With the quantum objects in hand, module `02_InformationAndGeometry` builds the information theory and geometry that classical and quantum optimal transport both stand on: Shannon entropy (`02/01` — where $H(A) \le H(AB)$ and the binary entropy $h(p)$ of `01/18` are proved), relative entropy, mutual information, the probability simplex and its two geometries, then their quantum versions — Umegaki relative entropy, quantum mutual information (`02/10`, where the coins and the Bell pair reappear), negative conditional entropy (`02/11`) and the Bures distance (`02/12`, with the fidelity convention of `01/14`). It opens with `02/01_shannon_entropy`.

## References

- M. A. Nielsen & I. L. Chuang, *Quantum Computation and Quantum Information*, §8.2 (operator-sum representation; environment models; §8.2.4 unitary freedom of Kraus operators), §8.3 (bit flip, phase flip, depolarizing, amplitude damping, phase damping; $T_1$, $T_2$), §9.2 (contractivity of the trace distance), Cambridge University Press (2010). doi:10.1017/CBO9780511976667
- W. F. Stinespring, *Positive functions on C*-algebras*, Proc. Amer. Math. Soc. **6**, 211 (1955) — every channel is "unitary on a larger system, then a partial trace". doi:10.1090/S0002-9939-1955-0069403-4
- M.-D. Choi, *Completely positive linear maps on complex matrices*, Linear Algebra Appl. **10**, 285 (1975) — complete positivity and the Kraus form. doi:10.1016/0024-3795(75)90075-0
- K. Kraus, *States, Effects, and Operations*, Lecture Notes in Physics **190**, Springer (1983). doi:10.1007/3-540-12732-1
- P. Krantz, M. Kjaergaard, F. Yan, T. P. Orlando, S. Gustavsson & W. D. Oliver, *A quantum engineer's guide to superconducting qubits*, Appl. Phys. Rev. **6**, 021318 (2019) — $T_1$, $T_2$, $T_\varphi$ and $1/T_2 = 1/2T_1 + 1/T_\varphi$ on real devices. doi:10.1063/1.5089550
- M. M. Wilde, *Quantum Information Theory*, 2nd ed., §4.4 (noisy evolution; Choi–Kraus theorem), Cambridge University Press (2017). doi:10.1017/9781316809976

**Previous:** `notebooks/01_QuantumFoundations/18_entanglement.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/01_shannon_entropy.ipynb`